# Занятие 11. Калибровка камеры

Курс «Введение в компьютерное зрение» · Innopolis University · Fall 2026.
Опора: **L11 «Модель камеры и калибровка»**, страницы PDF 39–45, 52–58.
Здесь и далее номера означают страницы файла, а не номера в подвале.

За 90 минут получим K и коэффициенты дисторсии, найдём испорченное наблюдение,
повторим калибровку, определим позу доски и нарисуем куб.
Три пропуска по 5–12 строк, после каждого — самопроверка; ⭐ необязательна.
Работаем в классе: 0–15 — опорный пример, 15–65 — TODO, 65–80 — разбор,
80–90 — мост к HW4. Критерии участия — по действующим договорённостям курса.

**Данные синтетические:** рендеры ChArUco с известной геометрией.
Камера, сеть и загрузки не нужны. Результаты не являются измерениями реальной камеры.
Чисел RMS этого набора в L11 нет; мы измеряем их отдельно.

## 0. Окружение
Выберите ядро окружения курса (`iu-intro-cv/.venv`). Если импорт не проходит,
установите зависимости из корня публичного репозитория: `python -m pip install -r requirements.txt`.
См. `docs/setup-guide.md`. Эта ячейка ничего не скачивает и не открывает камеру.

In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata
import cv2
import numpy as np
import matplotlib.pyplot as plt
import cvcourse

assert cv2.__version__ == '4.14.0', cv2.__version__
assert importlib.metadata.version('opencv-contrib-python') == '4.14.0.94'
assert hasattr(cv2.aruco, 'CharucoDetector'), 'Use opencv-contrib-python'
print('OpenCV', cv2.__version__, '| NumPy', np.__version__)

## 1. Данные и опорный пример

ChArUco — шахматка с ArUco-маркерами: идентификатор угла связывает пиксельную
координату с точкой доски даже при частичной видимости. Размер **7×5 клеток**,
внутренних углов **6×4**; клетка 0.03 m, маркер 0.021 m, DICT_4X4_50.
Объектные точки — метры в плоскости Z=0 доски, изображение — пиксели.
`image_size=(width,height)`, а `image.shape=(height,width)`.

Генератор сначала создаёт доску `CharucoBoard.generateImage`, затем для каждого
пикселя камеры обращает дисторсию и пересекает луч с плоскостью доски.
Одна гомография не моделировала бы ненулевую дисторсию объектива.
Изображения получаются из текстуры, **углы ниже действительно ищет детектор**.
K_GT и DIST_GT участвуют только в рендере и независимой самопроверке, не в калибровке.

In [ ]:
def make_scene(pattern='charuco'):
    size = (640, 480)
    K = np.array([[720., 0., 320.], [0., 700., 240.], [0., 0., 1.]])
    dist = np.array([-.16, .04, .001, -.0008, 0.])
    board = cv2.aruco.CharucoBoard((7, 5), .03, .021,
                                 cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50))
    texture = board.generateImage((1120, 800), marginSize=0, borderBits=1)
    if pattern == 'chessboard':
        sy, sx = np.mgrid[:800, :1120]
        texture = np.uint8(((sx // 160 + sy // 160) % 2) * 255)
    elif pattern != 'charuco':
        raise ValueError('Unknown pattern')
    yy, xx = np.mgrid[:size[1], :size[0]].astype(np.float32)
    pixels = np.stack([xx, yy], axis=-1).reshape(-1, 1, 2)
    rays = cv2.undistortPoints(pixels, K, dist).reshape(-1, 2)
    rays = np.column_stack([rays, np.ones(len(rays))])
    images, poses = [], []
    for i in range(20):
        angles = np.array([-.38 + .19 * (i % 5), -.40 + .26 * (i // 5), -.12 + .08 * (i % 4)])
        R = cv2.Rodrigues(angles)[0]
        center = np.array([-.095 + .0475 * (i % 5), -.07 + .046 * (i // 5), .46 + .025 * (i % 3)])
        t = center - R @ np.array([.105, .075, 0.])
        depth = np.dot(R[:, 2], t) / (rays @ R[:, 2])
        world = (depth[:, None] * rays - t) @ R
        mx = (world[:, 0].reshape(size[1], size[0]) / .21 * 1120 - .5).astype(np.float32)
        my = (world[:, 1].reshape(size[1], size[0]) / .15 * 800 - .5).astype(np.float32)
        image = cv2.remap(texture, mx, my, cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=190)
        images.append(image)
        poses.append((angles, t))
    return board, images, K, dist, size, poses

In [ ]:
board, synthetic_images, K_GT, DIST_GT, image_size, poses_gt = make_scene()
blank = np.full((image_size[1], image_size[0]), 190, np.uint8)
frames = synthetic_images + [blank]
synthetic = True
print('Poses:', len(synthetic_images), '| size (W,H):', image_size)
print('K_GT (px):\n', K_GT, '\nDIST_GT:', DIST_GT)
print('Images SHA-256:', hashlib.sha256(b''.join(x.tobytes() for x in frames)).hexdigest())
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, i in zip(axes, (0, 7, 19)):
    ax.imshow(frames[i], cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'View {i}'); ax.axis('off')
plt.tight_layout(); plt.show()

### Слот для своих фото (по желанию)

По умолчанию `OWN_DIR=None`: обязательная часть полностью автономна.
Если уже есть снимки **этой же ChArUco-доски**, укажите папку в ячейке ниже.
Используйте одно разрешение и фиксированный фокус; размер клетки должен
соответствовать физической доске. Для другой доски измените `board` вместе с её
размерами/словарём. PNG/JPG читаются без зависимости от текущего диска/кириллицы пути.
Новые снимки сейчас делать не требуется. На своих данных GT-проверки и учебное
повреждение координат отключаются. Порог отбора — диагностическая эвристика,
его нужно осмысленно выбирать для своего набора.

In [ ]:
OWN_DIR = None  # Example: Path('data/own')
if OWN_DIR is not None:
    paths = sorted(p for p in Path(OWN_DIR).iterdir() if p.suffix.lower() in ('.png', '.jpg', '.jpeg'))
    if not paths:
        raise ValueError('No PNG/JPG images in OWN_DIR')
    frames = [cv2.imdecode(np.fromfile(p, np.uint8), cv2.IMREAD_GRAYSCALE) for p in paths]
    if any(x is None for x in frames):
        raise ValueError('Unreadable image')
    image_size = (frames[0].shape[1], frames[0].shape[0])
    synthetic = False
    print('Own images:', len(frames), '| size:', image_size)

### Опорный пример: ошибка в пикселях

Модель: `X_camera = R @ X_board + t`, затем перспективная проекция с K и dist.
K — внутренние параметры; R,t меняются от кадра к кадру.
`projectPoints` возвращает `(N,1,2)` — перед вычитанием приведём оба массива к `(N,2)`.
RMS здесь — `sqrt(mean(sum((prediction-observation)**2, axis=1)))`.
Две ошибки (3,4) px дают RMS **5 px**, а не норму, делённую на число точек.

In [ ]:
example_error = np.array([[3., 4.], [3., 4.]])
example_rms = np.sqrt(np.mean(np.sum(example_error**2, axis=1)))
assert np.isclose(example_rms, 5.)
print('Point RMS:', example_rms, 'px')

## 2. TODO 1 — найти углы и собрать соответствия (11 строк)

Верните четыре списка: объектные точки, пиксели, индексы принятых и отклонённых кадров.
В цикле вызовите `CharucoDetector(board).detectBoard(frame)`: первые два выхода —
`corners`, `ids`. Пропускайте кадры с отсутствующими, малочисленными (<6) или
коллинеарными углами. `board.matchImagePoints(corners, ids)` возвращает правильные
3D↔2D пары; не создавайте сетку по числу найденных углов — при частичном обзоре
идентификаторы могут иметь пропуски. Готовые проверки размеров оставьте перед циклом.

In [ ]:
def collect_views(frames, board):
    if not frames or any(x is None or x.ndim != 2 or x.dtype != np.uint8 for x in frames):
        raise ValueError('Expected nonempty uint8 grayscale frames')
    if any(x.shape != frames[0].shape for x in frames):
        raise ValueError('All frames must have one resolution')
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 11 lines: detectBoard, reject unusable views, matchImagePoints
    raise NotImplementedError("TODO 1")
    # ───────────────────────────────────────────────────────────

In [ ]:
objects, observations, accepted, rejected = collect_views(frames, board)
assert len(objects) == len(observations) == len(accepted) >= 3
assert all(a.reshape(-1, 3).shape[0] == b.reshape(-1, 2).shape[0] for a, b in zip(objects, observations))
assert all(np.isfinite(x).all() for x in objects + observations)
if synthetic:
    assert accepted == list(range(20)) and rejected == [20]
    assert all(len(x) == 24 for x in observations)
print('Accepted:', accepted, '| rejected:', rejected)
view = cv2.cvtColor(frames[accepted[0]], cv2.COLOR_GRAY2BGR)
for xy in observations[0].reshape(-1, 2):
    cv2.circle(view, tuple(np.rint(xy).astype(int)), 3, (0, 0, 255), 1)
plt.figure(figsize=(7, 4)); plt.imshow(cv2.cvtColor(view, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()

## 3. TODO 2 — калибровка, per-image RMS, отбор и повтор (10 строк)

Для контролируемого эксперимента **после детекции** добавим шум координатам
кадра 7: σ=4 px, seed=11. Само изображение не размывается и не меняется;
это моделирование сбоя локализации, а не результат работы детектора.

`calibrateCameraExtended(objects,images,image_size,None,None)` оценивает K и dist
без GT-подсказок. Выходы 0–4: RMS,K,dist,rvecs,tvecs; последний — per-view RMS.
Сделайте один проход отбора `errors <= max(floor_px, factor * median(errors))`,
проверьте, что осталось хотя бы три позы, и заново оцените параметры на оставшихся.
Возвращайте словарь с `before`, `after`, `keep`, `errors_before`, `errors_after`.
Снижение RMS на другом поднаборе само по себе не доказывает улучшение камеры:
посмотрим также K и независимую контрольную позу в тестах.

In [ ]:
observed = [x.copy() for x in observations]
if synthetic:
    bad_position = accepted.index(7)
    noise = np.random.default_rng(11).normal(0, 4, observed[bad_position].shape).astype(np.float32)
    observed[bad_position] += noise

In [ ]:
def calibrate_trim(objects, images, image_size, floor_px=.6, factor=2.5):
    if len(objects) != len(images) or len(objects) < 3:
        raise ValueError('Need matching lists with at least three views')
    if floor_px <= 0 or factor <= 0 or min(image_size) <= 0:
        raise ValueError('Thresholds and image size must be positive')
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 10 lines: calibrateCameraExtended, per-view mask, recalibrate
    raise NotImplementedError("TODO 2")
    # ───────────────────────────────────────────────────────────

In [ ]:
calibration = calibrate_trim(objects, observed, image_size)
rms_before, K_before, dist_before, _, _ = calibration['before']
rms_after, K, dist, rvecs, tvecs = calibration['after']
keep = calibration['keep']
assert keep.dtype == bool and len(keep) == len(objects) and keep.sum() >= 3
assert np.isfinite(K).all() and K[0, 0] > 0 and K[1, 1] > 0
assert calibration['errors_after'].shape == (int(keep.sum()),)
if synthetic:
    assert np.flatnonzero(~keep).tolist() == [accepted.index(7)]
    assert .8 < rms_before < 1.4 and .15 < rms_after < .5
    assert np.max(np.abs(np.diag(K)[:2] / np.diag(K_GT)[:2] - 1)) < .02
print(f'RMS before / after: {rms_before:.6f} / {rms_after:.6f} px')
print('Rejected original views:', [accepted[i] for i in np.flatnonzero(~keep)])
print('K before:\n', K_before, '\nK after:\n', K, '\nK difference:\n', K - K_before)
print('dist:', dist.ravel())
fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(accepted, calibration['errors_before'], color=['steelblue' if k else 'tomato' for k in keep])
ax.set(xlabel='Original view index', ylabel='Point RMS (px)', title='Before trimming')
plt.tight_layout(); plt.show()

## 4. TODO 3 — поза доски и куб (9 строк)

`solvePnP` с оценёнными K/dist даёт **преобразование из доски в камеру**.
Куб задаётся в метрах; отрицательное Z вершины означает сторону перед доской,
ближе к камере. Нарисуем куб на исправленном изображении.
Нельзя наносить искажённые пиксели на `undistort`-кадр: переведите вершины
через `undistortPoints(..., P=K)`. Без P результат был бы в нормированных координатах.

Нужные операции: `solvePnP` → проверка успеха → `projectPoints` для куба и углов
→ point RMS → `undistort` → `undistortPoints` → готовая `draw_cube`.
Верните `rvec,tvec,rms,overlay,cube_pixels`; `cube_pixels` — на исправленном кадре.

In [ ]:
CUBE = np.float32([[.04,.04,0], [.10,.04,0], [.10,.10,0], [.04,.10,0],
                   [.04,.04,-.06], [.10,.04,-.06], [.10,.10,-.06], [.04,.10,-.06]])

def draw_cube(image, points):
    canvas = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    p = np.rint(points).astype(np.int32).reshape(8, 2)
    for a, b in ((0,1),(1,2),(2,3),(3,0),(4,5),(5,6),(6,7),(7,4),(0,4),(1,5),(2,6),(3,7)):
        cv2.line(canvas, tuple(p[a]), tuple(p[b]), (0, 170, 0) if a < 4 and b < 4 else (0, 0, 240), 2, cv2.LINE_AA)
    return canvas

def pose_cube(frame, objects, images, K, dist):
    if len(objects) < 6 or len(objects) != len(images):
        raise ValueError('Need at least six matching board points')
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 9 lines: solvePnP, projectPoints, point RMS, undistort and overlay
    raise NotImplementedError("TODO 3")
    # ───────────────────────────────────────────────────────────

In [ ]:
first_good = int(np.flatnonzero(keep)[0])
original_index = accepted[first_good]
pose = pose_cube(frames[original_index], objects[first_good], observed[first_good], K, dist)
assert pose['overlay'].shape == (*frames[original_index].shape, 3)
assert np.isfinite(pose['cube_pixels']).all() and pose['cube_pixels'].shape == (8, 2)
R = cv2.Rodrigues(pose['rvec'])[0]
assert ((CUBE @ R.T + pose['tvec'].reshape(3))[:, 2] > 0).all()
if synthetic:
    assert pose['rms'] < .6
    assert np.linalg.norm(pose['tvec'].ravel() - poses_gt[original_index][1]) < .02
print(f"PnP point RMS: {pose['rms']:.6f} px | t (m):", pose['tvec'].ravel())
print('Distance to board origin (m):', np.linalg.norm(pose['tvec']))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(frames[original_index], cmap='gray'); axes[0].set_title('Distorted frame')
axes[1].imshow(cv2.cvtColor(pose['overlay'], cv2.COLOR_BGR2RGB)); axes[1].set_title('Undistorted + cube')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

## 5. ⭐ Сравнение с обычной шахматкой

Сгенерируйте `make_scene(pattern='chessboard')`: та же камера, дисторсия и позы,
но текстура без маркеров. Найдите **6×4 внутренних угла** через
`findChessboardCornersSB`. Для каждого принятого кадра создайте grid XY с шагом
0.03 m и Z=0, соблюдая порядок углов. Оцените K/dist без искусственного выброса.

Сравните количество принятых кадров и ошибку fx/fy относительно GT, а затем RMS
двух методов на **общих принятых позах без добавленного шума**. Для такой сверки
вызовите калибровку заново на обоих поднаборах. Малый RMS на разных наборах
не определяет победителя. Объясните, что происходит с обычной шахматкой при
частичном обзоре. Код ⭐ можно написать в новой ячейке; обязательных TODO ровно три.

## 6. Мост к HW4 и вопросы для разбора

| Здесь | В `homeworks/hw4-calib3d/template/calib3d.py` |
|---|---|
| Объектные точки по ID, размеры в метрах | `board_points`, адаптация `detect_board` к ChArUco |
| K/dist и per-view RMS | `calibrate`, `reprojection_errors` |
| Отбраковка с сохранением индексов | `calibration_report` и сравнение K до/после |
| `solvePnP`, R,t и нормированные лучи | `pose`; затем своё пересечение с плоскостью в `plane_measure` |

Слот своих фото выше помогает проверить подготовленные данные; синтетика
не заменяет собственную калибровку и метрологию из ТЗ HW4.
Расписание, сроки и правила ДЗ берём из актуального README/Moodle.

1. Почему малая ошибка на одинаковых фронтальных позах не гарантирует верную K?
2. Что изменится в K и t, если перепутать метры и миллиметры размера клетки?
3. Почему нельзя смешивать `projectPoints` с dist и исправленное изображение?
4. Чем `t_z` отличается от расстояния `norm(t)` до начала координат доски?

Документация: [ChArUco detection](https://docs.opencv.org/4.12.0/df/d4a/tutorial_charuco_detection.html),
[calibration with ChArUco](https://docs.opencv.org/4.12.0/da/d13/tutorial_aruco_calibration.html).